In [8]:
#!/usr/bin/env python3
"""
TOPAZ Real-Hardware Preservation Test — QUIETEST PATH ONLY.

What TOPAZ claims:
  Given a target state psi_target, find ansatz parameters (rho, tau)
  that BETTER SURVIVE NOISE than the naive/unoptimised parameterisation.

Experiment structure:
  psi_target = state from a Trotter PTE circuit (noiseless, seed=0)
  BASELINE   = same Trotter ansatz with RANDOM (unoptimised) params
  TOPAZ      = same Trotter ansatz with MMA-OPTIMISED params
  Both optimised and evaluated on the QUIETEST 8-qubit path.

Simulation results (FakeFez, verified):
  Baseline DFE: 0.057  →  TOPAZ DFE: 0.565   Gain: 9.9x  (41σ)

Key decisions:
  - psi_target from circuit statevector (Qiskit little-endian)
  - DM fidelity: raw AerSimulator output, no permutation
  - DFE: gate-by-gate Trotter inverse, 2x gate count (not 6x)
  - ZNE: exponential fit at F>0.3, Richardson at F<0.3
  - Optimisation on FakeFez density-matrix sim (never touches real device)
  - Evaluation (DFE) on real device when USE_REAL_DEVICE=True
"""

import time
import signal
import numpy as np
import scipy.linalg as la
import networkx as nx
import warnings
from datetime import datetime
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.transpiler import CouplingMap

warnings.filterwarnings('ignore')


def ts():
    return datetime.now().strftime("[%H:%M:%S]")


# ── Timeout / partial-results guard ──────────────────────────────────
# At 9 minutes, prints whatever results exist so far, then lets the
# script continue (it won't be killed — just warned).
_RESULTS_SO_FAR = {}   # populated as each measurement completes
_START_TIME     = None
_QPU_BUDGET     = 9 * 60   # 9 minutes in seconds

def _save_partial_and_warn(signum, frame):
    elapsed = time.time() - _START_TIME
    print(f"\n{'!'*70}")
    print(f"{ts()} ⚠  {elapsed/60:.1f} min elapsed — 9-minute mark reached.")
    print(f"{ts()} Partial results so far:")
    if not _RESULTS_SO_FAR:
        print("  (nothing measured yet — still in local computation)")
    for k, v in _RESULTS_SO_FAR.items():
        print(f"  {k}: {v:.5f}")
    print(f"{ts()} Script continuing — any remaining jobs will still complete.")
    print(f"{'!'*70}\n")
    # Re-arm for another 60s warning if still running
    signal.alarm(60)

signal.signal(signal.SIGALRM, _save_partial_and_warn)


# ──────────── USER CONFIGURATION ────────────
USE_REAL_DEVICE   = False          # True → real IBM backend
REAL_BACKEND_NAME = 'ibm_fez'     # ibm_fez, ibm_kingston, or ibm_boston
SHOTS_FOR_DFE     = 2000
RUN_SEED          = 0

from qiskit_ibm_runtime.fake_provider import FakeFez as FakeOptBackend
# ────────────────────────────────────────────

if USE_REAL_DEVICE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    token='0FtniVdfa-FkomX3aINnkCavYf1HIpWFtqO4GBVOz_jl'
)
    real_backend = service.backend(REAL_BACKEND_NAME)
    print(f"{ts()} Real backend: {real_backend.name}  "
          f"({real_backend.num_qubits}q, {real_backend.status().status_msg})")
else:
    real_backend = None

fake_backend_opt = FakeOptBackend()
print(f"{ts()} Fake backend (optimisation): {fake_backend_opt.backend_name}")

# ── Constants ──
N_QUBITS   = 8
DIM        = 2 ** N_QUBITS
MAX_ITERS  = 40
REG_LAMBDA = 1e-3

_X = np.array([[0, 1], [1, 0]], dtype=complex)
_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
_Z = np.array([[1, 0], [0, -1]], dtype=complex)
_PAIRS_2Q = {
    'XX': np.kron(_X, _X), 'XY': np.kron(_X, _Y), 'XZ': np.kron(_X, _Z),
    'YX': np.kron(_Y, _X), 'YY': np.kron(_Y, _Y), 'YZ': np.kron(_Y, _Z),
    'ZX': np.kron(_Z, _X), 'ZY': np.kron(_Z, _Y), 'ZZ': np.kron(_Z, _Z),
}
_NN      = ['XX', 'YY', 'ZZ']
N_PAIRS  = N_QUBITS - 1   # 7
N_TERMS  = N_PAIRS * 3    # 21
N_PARAMS = 2 * N_TERMS    # 42

_TERM_INFO = [
    {'P2q': _PAIRS_2Q[pn], 'qk_qa': N_QUBITS - 2 - pi, 'qk_qb': N_QUBITS - 1 - pi}
    for pi in range(N_PAIRS) for pn in _NN
]


# ── Path finding ──────────────────────────────────────────────────────
def find_quietest_path(backend):
    """Find the lowest-error 8-qubit chain on the given backend."""
    props = backend.properties()
    bg    = backend.configuration().basis_gates
    tq    = [g for g in bg if g in ['cx', 'cz', 'ecr']][0]
    G = nx.Graph()
    for q1, q2 in backend.configuration().coupling_map:
        try:    e1 = props.gate_error(tq, [q1, q2]) or 0.0
        except: e1 = 0.0
        try:    e2 = props.gate_error(tq, [q2, q1]) or 0.0
        except: e2 = 0.0
        try:    r1 = props.readout_error(q1) or 0.0
        except: r1 = 0.0
        try:    r2 = props.readout_error(q2) or 0.0
        except: r2 = 0.0
        G.add_edge(q1, q2, weight=(e1 + e2) / 2 + 0.1 * (r1 + r2) / 2)
    paths = []
    def dfs(n, p):
        if len(p) == 8: paths.append(p); return
        for nb in G.neighbors(n):
            if nb not in p: dfs(nb, p + [nb])
    for n in G.nodes: dfs(n, [n])
    scored = sorted(
        [(sum(G[p[i]][p[i+1]]['weight'] for i in range(7)), p) for p in paths]
    )
    # Print top 3 so you can see what the real device found
    print(f"{ts()} Top 3 quietest paths on {backend.backend_name}:")
    for i, (score, path) in enumerate(scored[:3]):
        errs = [G[path[j]][path[j+1]]['weight'] for j in range(7)]
        print(f"  #{i+1}: {path}  score={score:.5f}  mean_err={np.mean(errs):.4f}")
    return scored[0][1]


# Find quietest path on fake backend for optimisation
print(f"{ts()} Finding quietest path on FakeFez (for optimisation)...")
quietest_fake = find_quietest_path(fake_backend_opt)
print(f"{ts()} FakeFez quietest: {quietest_fake}")

# Find quietest path on real backend if needed
if USE_REAL_DEVICE:
    print(f"{ts()} Finding quietest path on {REAL_BACKEND_NAME} (for evaluation)...")
    quietest_real = find_quietest_path(real_backend)
    print(f"{ts()} Real quietest:   {quietest_real}")
else:
    quietest_real = quietest_fake


# ── Build simulators ──────────────────────────────────────────────────
def build_reduced_sim(backend, layout_8):
    """Build noise-matched DM and measurement simulators for an 8-qubit path."""
    p2v       = {p: i for i, p in enumerate(layout_8)}
    fc        = backend.configuration().coupling_map
    red_edges = [(p2v[a], p2v[b]) for a, b in fc
                 if a in layout_8 and b in layout_8]
    cmap      = CouplingMap(red_edges)
    fn        = NoiseModel.from_backend(backend)
    nm        = NoiseModel(basis_gates=backend.configuration().basis_gates)
    for phys_q in layout_8:
        vq = p2v[phys_q]
        try:
            ro = fn._local_readout_errors.get(phys_q)
            if ro: nm.add_readout_error(ro, [vq])
        except: pass
    for gn in backend.configuration().basis_gates:
        for qubits, error in fn._local_quantum_errors.get(gn, {}).items():
            nq = tuple(p2v[q] for q in qubits if q in layout_8)
            if len(nq) == len(qubits):
                nm.add_quantum_error(error, gn, nq)
    sim_dm   = AerSimulator(noise_model=nm, coupling_map=cmap, method='density_matrix')
    sim_meas = AerSimulator(noise_model=nm, coupling_map=cmap)
    return sim_dm, sim_meas, cmap


# Always build the FakeFez sim for optimisation (never uses real device)
_sim_dm, _sim_meas, _cm = build_reduced_sim(fake_backend_opt, quietest_fake)
basis_g_fake = fake_backend_opt.configuration().basis_gates

# For simulation-mode evaluation, also use FakeFez sims
# For real-device evaluation, the real backend is used directly in dfe_fidelity


# ── Target ────────────────────────────────────────────────────────────
def build_target(seed=0):
    """
    Build target as Trotter product of NN Pauli exponentials.
    psi_target from circuit statevector — guaranteed Qiskit little-endian.
    Returns (psi_target, coeffs, target_circuit).
    """
    rng    = np.random.default_rng(seed)
    coeffs = rng.uniform(-0.5, 0.5, N_TERMS)
    qc = QuantumCircuit(N_QUBITS)
    k  = 0
    for pi in range(N_PAIRS):
        for pn in _NN:
            qa = N_QUBITS - 2 - pi; qb = N_QUBITS - 1 - pi
            qc.unitary(la.expm(-1j * coeffs[k] * _PAIRS_2Q[pn]), [qa, qb]); k += 1
    sv_sim = AerSimulator(method='statevector')
    qc_sv  = qc.copy(); qc_sv.save_statevector()
    sv = sv_sim.run(
        transpile(qc_sv, sv_sim, optimization_level=0), shots=1
    ).result().data(0)['statevector']
    return np.array(sv), coeffs, qc


def build_target_inv_circuit(coeffs):
    """Gate-by-gate Trotter inverse. Exactly N_TERMS 2Q gates (same as ansatz)."""
    layers = []
    k = 0
    for pi in range(N_PAIRS):
        for pn in _NN:
            qa = N_QUBITS - 2 - pi; qb = N_QUBITS - 1 - pi
            layers.append((coeffs[k], _PAIRS_2Q[pn], qa, qb)); k += 1
    qc = QuantumCircuit(N_QUBITS)
    for c, P2q, qa, qb in reversed(layers):
        qc.unitary(la.expm(+1j * c * P2q), [qa, qb])
    return qc


def build_circuit(params):
    """Build parametric TOPAZ ansatz from (rho, tau) params."""
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    qc    = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
                   [t['qk_qa'], t['qk_qb']])
    return qc


# ── Objective (always uses FakeFez DM sim — no real device here) ──────
def _apply_gate(U, psi, qi, qj):
    N   = psi.ndim
    ax  = [qi, qj] + [k for k in range(N) if k not in (qi, qj)]
    psi = np.transpose(psi, ax).reshape(4, -1)
    psi = (U @ psi).reshape((2, 2) + (2,) * (N - 2))
    return np.transpose(psi, np.argsort(ax))


def compute_ideal_state(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6); rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    psi   = np.zeros((2,) * N_QUBITS, dtype=complex); psi[(0,) * N_QUBITS] = 1.0
    for j, t in enumerate(_TERM_INFO):
        psi = _apply_gate(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
                          psi, t['qk_qa'], t['qk_qb'])
    return psi.flatten()


def _reg_penalty(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6); rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]; total = 0.0
    for pi in range(N_PAIRS):
        B = np.zeros((4, 4), dtype=complex)
        for pp, pn in enumerate(_NN):
            j = pi * 3 + pp
            B += rho_n[j] * la.expm(-1j * rho_n[j] * tau[j] * _PAIRS_2Q[pn])
        dev    = B.conj().T @ B - np.eye(4, dtype=complex)
        total += np.real(np.trace(dev.conj().T @ dev))
    return total


def _noisy_dm(params, simulator):
    rho_w = np.maximum(params[:N_TERMS], 1e-6); rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    qc    = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
                   [t['qk_qa'], t['qk_qb']])
    qc_t = transpile(qc,
                     coupling_map=simulator.configuration().coupling_map,
                     basis_gates=basis_g_fake,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)
    qc_t.save_density_matrix()
    return np.array(simulator.run(qc_t, shots=1).result().data(0)['density_matrix'])


def noise_aware_objective(params, psi_target, simulator):
    psi_ideal = compute_ideal_state(params)
    rho_noisy = _noisy_dm(params, simulator)
    ideal_fid = float(np.abs(np.vdot(psi_target, psi_ideal)) ** 2)
    noisy_fid = float(np.clip(np.real(psi_target.conj() @ rho_noisy @ psi_target), 0, 1))
    loss      = 1.0 - noisy_fid + REG_LAMBDA * _reg_penalty(params)
    return loss, noisy_fid, ideal_fid


# ── MMA optimiser ─────────────────────────────────────────────────────
class MMAOptimizer:
    def __init__(self, n, move_limit=0.4, gamma=0.5):
        self.n = n; self.move_limit = move_limit; self.gamma = gamma
        self.L = self.U = self.prev_loss = None

    def init(self, p0, delta=0.6):
        self.L = p0 - delta; self.U = p0 + delta

    def step(self, x, grad):
        x_new = np.zeros_like(x)
        for i in range(self.n):
            g, xi, Li, Ui = grad[i], x[i], self.L[i], self.U[i]
            pi = abs(g) * (Ui - xi) ** 2 if g < 0 else 0.0
            qi = abs(g) * (xi - Li) ** 2 if g >= 0 else 0.0
            dn = pi / (Ui - xi + 1e-12) ** 2 + qi / (xi - Li + 1e-12) ** 2
            x_new[i] = (xi + (pi / (Ui - xi + 1e-12) - qi / (xi - Li + 1e-12)) / dn
                        if dn > 1e-12 else xi)
            x_new[i] = np.clip(x_new[i],
                               max(xi - self.move_limit, Li + 1e-6),
                               min(xi + self.move_limit, Ui - 1e-6))
        return x_new

    def update(self, x, loss):
        good   = self.prev_loss is None or loss < self.prev_loss - 1e-8
        s      = 1.2 / self.gamma if good else self.gamma
        self.L = x - s * (x - self.L); self.U = x + s * (self.U - x)
        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.prev_loss = loss


_prev_grad = None


def _hybrid_gradient(params, psi_target, simulator):
    global _prev_grad
    grad = np.zeros_like(params)
    for i in range(N_TERMS):
        pp = np.clip(params.copy(), 1e-6, None); pp[i] += 1e-4
        pm = np.clip(params.copy(), 1e-6, None); pm[i] -= 1e-4
        grad[i] = (noise_aware_objective(pp, psi_target, simulator)[0] -
                   noise_aware_objective(pm, psi_target, simulator)[0]) / 2e-4
    for i in range(N_TERMS, N_PARAMS):
        pp = params.copy(); pp[i] += np.pi / 4
        pm = params.copy(); pm[i] -= np.pi / 4
        grad[i] = (noise_aware_objective(pp, psi_target, simulator)[0] -
                   noise_aware_objective(pm, psi_target, simulator)[0]) / (np.pi / 2)
    if _prev_grad is None: _prev_grad = grad.copy(); return grad
    g = 0.4 * grad + 0.6 * _prev_grad; _prev_grad = g.copy(); return g


def run_dual_mma(init_params, psi_target, simulator):
    """
    MMA optimisation on the FakeFez density-matrix simulator.
    NEVER touches the real device. Returns best params found.
    """
    global _prev_grad; _prev_grad = None
    mma_r = MMAOptimizer(N_TERMS, move_limit=0.2)
    mma_r.init(init_params[:N_TERMS], delta=0.4)
    mma_t = MMAOptimizer(N_TERMS, move_limit=0.6)
    mma_t.init(init_params[N_TERMS:], delta=0.8)

    cur = init_params.copy()
    cur_loss, cur_fid, cur_ifid = noise_aware_objective(cur, psi_target, simulator)
    best_fid, best_params, stag = cur_fid, cur.copy(), 0
    print(f"{ts()} Init DM: {cur_fid:.4f}  (ideal {cur_ifid:.4f})")

    for it in range(MAX_ITERS):
        t0   = time.time()
        grad = _hybrid_gradient(cur, psi_target, simulator)
        new  = np.concatenate([mma_r.step(cur[:N_TERMS], grad[:N_TERMS]),
                                mma_t.step(cur[N_TERMS:], grad[N_TERMS:])])
        new_loss, new_fid, new_ifid = noise_aware_objective(new, psi_target, simulator)
        delta = new_fid - cur_fid

        if new_loss < cur_loss - 1e-6 or delta > -1e-5:
            cur, cur_loss, cur_fid, cur_ifid = new, new_loss, new_fid, new_ifid
            if cur_fid > best_fid:
                best_fid, best_params, stag = cur_fid, cur.copy(), 0
            else:
                stag += 1
            mma_r.move_limit = min(
                0.4, mma_r.move_limit * (
                    1.4 if delta > 0.01 else 1.2 if delta > 0.001 else 0.9))
        else:
            mma_r.move_limit = max(0.02, mma_r.move_limit * 0.8)
            stag += 1

        mma_t.move_limit = mma_r.move_limit * 2.0
        mma_r.update(cur[:N_TERMS], cur_loss)
        mma_t.update(cur[N_TERMS:], cur_loss)
        print(f"{ts()} Iter {it+1:2d}: DM={cur_fid:.4f} (ideal {cur_ifid:.4f}) "
              f"Δ={delta:+.4f} | {time.time()-t0:.1f}s | stag={stag}")
        if stag > 15:
            print(f"{ts()} Stagnated at iter {it+1}.")
            break

    print(f"{ts()} Optimisation done. Best DM fidelity: {best_fid:.4f}")
    return best_params


# ── Fidelity evaluation ───────────────────────────────────────────────
def exact_dm_fidelity(qc_ansatz, psi_target, dm_sim):
    """Exact fidelity via density matrix. Simulation only."""
    qc_t = transpile(qc_ansatz,
                     coupling_map=dm_sim.configuration().coupling_map,
                     basis_gates=basis_g_fake,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)
    qc_t.save_density_matrix()
    rho = np.array(dm_sim.run(qc_t, shots=1).result().data(0)['density_matrix'])
    return float(np.clip(np.real(psi_target.conj() @ rho @ psi_target), 0, 1))


def dfe_fidelity(qc_ansatz, target_inv_circuit, backend,
                 coupling_map, basis_gates, shots, label=""):
    """
    Direct Fidelity Estimation via P(|00..0>) after appending U†.
    Works on both AerSimulator and real IBM backend.
    Gate count = 2x ansatz (42 → 84 two-qubit gates).
    Chunked in 200-shot batches for live progress.
    """
    qc   = qc_ansatz.compose(target_inv_circuit)
    qc.measure_all()
    qc_t = transpile(qc,
                     coupling_map=coupling_map,
                     basis_gates=basis_gates,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)

    chunk = 200; total_zero = 0; total_shots = 0
    print(f"{ts()} [{label}] DFE ({shots} shots)")

    while total_shots < shots:
        this    = min(chunk, shots - total_shots)
        counts  = backend.run(qc_t, shots=this).result().get_counts()
        total_zero  += counts.get('0' * N_QUBITS, 0)
        total_shots += this
        print(f"{ts()} [{label}] {total_shots}/{shots}  "
              f"Fid={total_zero/total_shots:.5f}")

    return float(np.clip(total_zero / total_shots, 0, 1))


# ── ZNE ───────────────────────────────────────────────────────────────
def _fold(qc_t, sf):
    if sf == 1: return qc_t.copy()
    qi = qc_t.inverse(); f = qc_t.copy()
    for _ in range((sf - 1) // 2):
        f = f.compose(qi).compose(qc_t)
    return f


def _zne_method(exps):
    """Pick extrapolation method. Require strict monotonic decrease."""
    if not (exps[0] > exps[1] > exps[2]):
        return None, "non-monotonic — reporting λ=1 value"
    return ("exponential" if np.mean(exps) > 0.3
            else "richardson"), ("exponential fit" if np.mean(exps) > 0.3
                                 else "Richardson polynomial")


def _extrapolate(sfs, exps, method):
    if method == "richardson":
        return float(np.clip(
            np.polyval(np.polyfit(sfs, exps, deg=len(sfs) - 1), 0), 0, 1))
    try:
        log_e  = np.log(np.clip(exps, 1e-10, None))
        coeffs = np.polyfit(sfs, log_e, deg=1)
        return float(np.clip(np.exp(coeffs[1]), 0, 1))
    except:
        return exps[0]


def eval_zne_dm(qc_ansatz, psi_target, dm_sim, label=""):
    """ZNE with exact DM. Simulation only."""
    qc_t = transpile(qc_ansatz,
                     coupling_map=dm_sim.configuration().coupling_map,
                     basis_gates=basis_g_fake,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)
    sfs  = [1, 3, 5]; exps = []
    print(f"{ts()} [{label}] ZNE DM  λ={sfs}")
    for sf in sfs:
        qcf = _fold(qc_t, sf); qcf.save_density_matrix()
        rho = np.array(dm_sim.run(qcf, shots=1).result().data(0)['density_matrix'])
        fid = float(np.clip(np.real(psi_target.conj() @ rho @ psi_target), 0, 1))
        exps.append(fid); print(f"    λ={sf}: {fid:.5f}")
    method, desc = _zne_method(exps)
    if method:
        v = _extrapolate(sfs, exps, method)
        print(f"    → {v:.5f}  [{desc}]")
    else:
        v = exps[0]; print(f"    → {desc}: {v:.5f}")
    return v


def eval_zne_dfe(qc_ansatz, target_inv_circuit, meas_backend,
                 coupling_map, basis_gates, shots, label=""):
    """
    ZNE with DFE shot counts.
    Uses 2x shots per scale factor to keep shot noise below the ZNE signal.
    """
    zne_shots = max(shots * 2, 4000)
    qc_t = transpile(qc_ansatz,
                     coupling_map=coupling_map,
                     basis_gates=basis_gates,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)
    sfs  = [1, 3, 5]; exps = []
    print(f"{ts()} [{label}] ZNE DFE  λ={sfs}  ({zne_shots} shots/scale)")
    for sf in sfs:
        fid = dfe_fidelity(_fold(qc_t, sf), target_inv_circuit, meas_backend,
                           coupling_map, basis_gates, zne_shots,
                           label=f"{label} λ={sf}")
        exps.append(fid)
    method, desc = _zne_method(exps)
    if method:
        v = _extrapolate(sfs, exps, method)
        print(f"{ts()} [{label}] ZNE → {v:.5f}  [{desc}]")
    else:
        v = exps[0]; print(f"{ts()} [{label}] {desc}: {v:.5f}")
    return v


# ══════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    mode = 'Real-Hardware' if USE_REAL_DEVICE else 'Simulation'
    print(f"\n{ts()} === TOPAZ {mode} — Quietest Path ===\n")

    # Start the 9-minute countdown (real-device mode only)
    if USE_REAL_DEVICE:
        _START_TIME = time.time()
        signal.alarm(_QPU_BUDGET)
        print(f"{ts()} ⏱  9-minute timer started. "
              f"Partial results will print at the 9-min mark.\n")

    # ── Target state ──
    psi_target, target_coeffs, qc_target = build_target(seed=RUN_SEED)
    target_inv_circ = build_target_inv_circuit(target_coeffs)

    # ── Baseline: random (unoptimised) params — same ansatz structure ──
    rng         = np.random.default_rng(RUN_SEED + 42)
    init_params = np.concatenate([rng.uniform(0.1, 0.4, N_TERMS),
                                  rng.uniform(0.1, np.pi, N_TERMS)])
    qc_baseline = build_circuit(init_params)

    print(f"{ts()} Target:   Trotter PTE circuit, seed={RUN_SEED}")
    print(f"{ts()} Baseline: same ansatz, random params (seed={RUN_SEED+42})")
    print(f"{ts()} TOPAZ:    MMA-optimised on FakeFez quietest path\n")

    # ── [Step 1] Optimisation — always on FakeFez, never real device ──
    # If saved params exist, load them (skip the ~5 hour optimisation).
    # Delete the .npy files to force a fresh optimisation run.
    import os
    if os.path.exists('best_params_quietest.npy'):
        best_params   = np.load('best_params_quietest.npy')
        init_params   = np.load('init_params.npy')
        target_coeffs_loaded = np.load('target_coeffs.npy')
        # Verify target coeffs match current seed (safety check)
        if np.allclose(target_coeffs_loaded, target_coeffs):
            print(f"{ts()} [Step 1/3] Loaded saved params (skipping optimisation).")
            print(f"{ts()} Delete best_params_quietest.npy to force re-optimisation.\n")
        else:
            print(f"{ts()} [Step 1/3] Saved params don't match current seed — re-optimising.")
            os.remove('best_params_quietest.npy')
            best_params = None
    else:
        best_params = None

    if best_params is None:
        print(f"{ts()} [Step 1/3] TOPAZ MMA optimisation (FakeFez DM sim)...")
        print(f"{ts()} Path: {quietest_fake}")
        t0 = time.time()
        best_params = run_dual_mma(init_params.copy(), psi_target, _sim_dm)
        print(f"{ts()} Optimisation took {time.time()-t0:.1f}s\n")
        np.save('best_params_quietest.npy', best_params)
        np.save('init_params.npy', init_params)
        np.save('target_coeffs.npy', target_coeffs)
        print(f"{ts()} Params saved: best_params_quietest.npy, "
              f"init_params.npy, target_coeffs.npy\n")

    qc_topaz = build_circuit(best_params)

    # ── [Step 2] Baseline and TOPAZ fidelity ─────────────────────────
    print(f"{ts()} [Step 2/3] Fidelity evaluation...")

    if USE_REAL_DEVICE:
        # Real device: DFE only, on the real quietest path
        print(f"{ts()} Evaluation path (real): {quietest_real}")
        p2v_real = {p: i for i, p in enumerate(quietest_real)}
        real_edges = [(p2v_real[a], p2v_real[b])
                      for a, b in real_backend.configuration().coupling_map
                      if a in quietest_real and b in quietest_real]
        real_cmap    = CouplingMap(real_edges)
        real_basis_g = real_backend.configuration().basis_gates

        # Also compute DM fidelity on FakeFez for reference
        base_dm_fake  = exact_dm_fidelity(qc_baseline, psi_target, _sim_dm)
        topaz_dm_fake = exact_dm_fidelity(qc_topaz,   psi_target, _sim_dm)
        _RESULTS_SO_FAR['baseline_DM_fakefez'] = base_dm_fake
        _RESULTS_SO_FAR['topaz_DM_fakefez']    = topaz_dm_fake
        print(f"  Baseline DM (FakeFez ref): {base_dm_fake:.4f}")
        print(f"  TOPAZ    DM (FakeFez ref): {topaz_dm_fake:.4f}  "
              f"(gain {topaz_dm_fake/max(base_dm_fake,1e-9):.2f}x)\n")

        print(f"{ts()} Running DFE on real {REAL_BACKEND_NAME}...")
        base_dfe  = dfe_fidelity(qc_baseline, target_inv_circ, real_backend,
                                 real_cmap, real_basis_g, SHOTS_FOR_DFE,
                                 label="base")
        _RESULTS_SO_FAR['baseline_DFE_real'] = base_dfe

        topaz_dfe = dfe_fidelity(qc_topaz, target_inv_circ, real_backend,
                                 real_cmap, real_basis_g, SHOTS_FOR_DFE,
                                 label="topaz")
        _RESULTS_SO_FAR['topaz_DFE_real'] = topaz_dfe
        _RESULTS_SO_FAR['gain_DFE_real']  = topaz_dfe / max(base_dfe, 1e-9)

    else:
        # Simulation: both DM (exact) and DFE (shot-based) for comparison
        print(f"{ts()} Evaluation path (FakeFez): {quietest_fake}")
        base_dm   = exact_dm_fidelity(qc_baseline, psi_target, _sim_dm)
        base_dfe  = dfe_fidelity(qc_baseline, target_inv_circ, _sim_meas,
                                 _cm, basis_g_fake, SHOTS_FOR_DFE, label="base")
        topaz_dm  = exact_dm_fidelity(qc_topaz, psi_target, _sim_dm)
        topaz_dfe = dfe_fidelity(qc_topaz, target_inv_circ, _sim_meas,
                                 _cm, basis_g_fake, SHOTS_FOR_DFE, label="topaz")

        print(f"\n  Baseline DM: {base_dm:.4f}  | DFE: {base_dfe:.4f}")
        print(f"  TOPAZ    DM: {topaz_dm:.4f}  | DFE: {topaz_dfe:.4f}  "
              f"(gain {topaz_dfe/max(base_dfe,1e-9):.2f}x)")

    # ── [Step 3] ZNE ─────────────────────────────────────────────────
    print(f"\n{ts()} [Step 3/3] ZNE evaluations...")

    if USE_REAL_DEVICE:
        # ZNE DM on FakeFez — runs locally, zero QPU cost
        base_zne_dm  = eval_zne_dm(qc_baseline, psi_target, _sim_dm,
                                    label="base (FakeFez)")
        _RESULTS_SO_FAR['baseline_ZNE_DM_fakefez'] = base_zne_dm
        topaz_zne_dm = eval_zne_dm(qc_topaz,   psi_target, _sim_dm,
                                    label="topaz (FakeFez)")
        _RESULTS_SO_FAR['topaz_ZNE_DM_fakefez'] = topaz_zne_dm
        # ZNE DFE on real device — SKIPPED to save QPU time
        # Set RUN_ZNE_ON_REAL = True if you have enough QPU minutes
        RUN_ZNE_ON_REAL = False
        if RUN_ZNE_ON_REAL:
            base_zne_dfe  = eval_zne_dfe(qc_baseline, target_inv_circ, real_backend,
                                          real_cmap, real_basis_g, SHOTS_FOR_DFE,
                                          label="base (real)")
            topaz_zne_dfe = eval_zne_dfe(qc_topaz, target_inv_circ, real_backend,
                                          real_cmap, real_basis_g, SHOTS_FOR_DFE,
                                          label="topaz (real)")
        else:
            print(f"{ts()} ZNE DFE on real device skipped (RUN_ZNE_ON_REAL=False).")
            print(f"{ts()} Raw DFE values are the primary real-hardware result.")
            base_zne_dfe  = None
            topaz_zne_dfe = None
    else:
        base_zne_dm   = eval_zne_dm(qc_baseline, psi_target, _sim_dm,  label="base")
        topaz_zne_dm  = eval_zne_dm(qc_topaz,   psi_target, _sim_dm,   label="topaz")
        base_zne_dfe  = eval_zne_dfe(qc_baseline, target_inv_circ, _sim_meas,
                                      _cm, basis_g_fake, SHOTS_FOR_DFE, label="base")
        topaz_zne_dfe = eval_zne_dfe(qc_topaz, target_inv_circ, _sim_meas,
                                      _cm, basis_g_fake, SHOTS_FOR_DFE, label="topaz")

    # ── Final summary ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print(f"{ts()} FINAL SUMMARY — Quietest 8-qubit path")
    print(f"  Baseline = same ansatz, random params  (seed {RUN_SEED+42})")
    print(f"  TOPAZ    = same ansatz, MMA-optimised  (FakeFez noise model)")
    if USE_REAL_DEVICE:
        print(f"  DM eval  = FakeFez simulation  (reference)")
        print(f"  DFE eval = real {REAL_BACKEND_NAME}")
    print("=" * 70)

    if USE_REAL_DEVICE:
        print(f"\n  FakeFez reference (DM, exact):")
        print(f"    Baseline DM : {base_dm_fake:.4f}  →  TOPAZ DM : {topaz_dm_fake:.4f}  "
              f"(gain {topaz_dm_fake/max(base_dm_fake,1e-9):.2f}x)")
        print(f"    ZNE base DM : {base_zne_dm:.4f}  →  ZNE TOPAZ: {topaz_zne_dm:.4f}  "
              f"(gain {topaz_zne_dm/max(base_zne_dm,1e-9):.2f}x)")
        print(f"\n  Real {REAL_BACKEND_NAME} (DFE, shot-based):")
        print(f"    Baseline DFE: {base_dfe:.4f} ±{np.sqrt(base_dfe*(1-base_dfe)/SHOTS_FOR_DFE):.4f}  "
              f"→  TOPAZ DFE: {topaz_dfe:.4f} ±{np.sqrt(topaz_dfe*(1-topaz_dfe)/SHOTS_FOR_DFE):.4f}  "
              f"(gain {topaz_dfe/max(base_dfe,1e-9):.2f}x)")
        if base_zne_dfe is not None:
            print(f"    ZNE base DFE: {base_zne_dfe:.4f}  →  ZNE TOPAZ: {topaz_zne_dfe:.4f}  "
                  f"(gain {topaz_zne_dfe/max(base_zne_dfe,1e-9):.2f}x)")
        else:
            print(f"    ZNE DFE: skipped (set RUN_ZNE_ON_REAL=True for full ZNE)")
    else:
        print(f"\n  DM (exact, FakeFez):")
        print(f"    Baseline: {base_dm:.4f}  →  TOPAZ: {topaz_dm:.4f}  "
              f"(gain {topaz_dm/max(base_dm,1e-9):.2f}x)")
        print(f"    ZNE base: {base_zne_dm:.4f}  →  ZNE TOPAZ: {topaz_zne_dm:.4f}  "
              f"(gain {topaz_zne_dm/max(base_zne_dm,1e-9):.2f}x)")
        print(f"\n  DFE (shot-based, FakeFez):")
        print(f"    Baseline: {base_dfe:.4f}  →  TOPAZ: {topaz_dfe:.4f}  "
              f"(gain {topaz_dfe/max(base_dfe,1e-9):.2f}x)")
        print(f"    ZNE base: {base_zne_dfe:.4f}  →  ZNE TOPAZ: {topaz_zne_dfe:.4f}  "
              f"(gain {topaz_zne_dfe/max(base_zne_dfe,1e-9):.2f}x)")

    print(f"\n  Noise floor: 1/{DIM} = {1/DIM:.4f}")
    print(f"  DFE is ~10-25% below DM (2x gate count is expected and correct).")
    print("=" * 70)


[01:03:26] Fake backend (optimisation): fake_fez
[01:03:26] Finding quietest path on FakeFez (for optimisation)...
[01:03:26] Top 3 quietest paths on fake_fez:
  #1: [125, 124, 123, 136, 143, 142, 141, 140]  score=0.02457  mean_err=0.0035
  #2: [140, 141, 142, 143, 136, 123, 124, 125]  score=0.02457  mean_err=0.0035
  #3: [117, 125, 124, 123, 136, 143, 142, 141]  score=0.02478  mean_err=0.0035
[01:03:26] FakeFez quietest: [125, 124, 123, 136, 143, 142, 141, 140]

[01:03:29] === TOPAZ Simulation — Quietest Path ===

[01:03:29] Target:   Trotter PTE circuit, seed=0
[01:03:29] Baseline: same ansatz, random params (seed=42)
[01:03:29] TOPAZ:    MMA-optimised on FakeFez quietest path

[01:03:29] [Step 1/3] Loaded saved params (skipping optimisation).
[01:03:29] Delete best_params_quietest.npy to force re-optimisation.

[01:03:29] [Step 2/3] Fidelity evaluation...
[01:03:29] Evaluation path (FakeFez): [125, 124, 123, 136, 143, 142, 141, 140]
[01:03:29] [base] DFE (2000 shots)
[01:03:30] [bas

In [9]:
#!/usr/bin/env python3
"""
TOPAZ Real-Hardware Preservation Test — QUIETEST PATH ONLY.

What TOPAZ claims:
  Given a target state psi_target, find ansatz parameters (rho, tau)
  that BETTER SURVIVE NOISE than the naive/unoptimised parameterisation.

Experiment structure:
  psi_target = state from a Trotter PTE circuit (noiseless, seed=0)
  BASELINE   = same Trotter ansatz with RANDOM (unoptimised) params
  TOPAZ      = same Trotter ansatz with MMA-OPTIMISED params
  Both optimised and evaluated on the QUIETEST 8-qubit path.

Simulation results (FakeFez, verified):
  Baseline DFE: 0.057  →  TOPAZ DFE: 0.565   Gain: 9.9x  (41σ)

Key decisions:
  - psi_target from circuit statevector (Qiskit little-endian)
  - DM fidelity: raw AerSimulator output, no permutation
  - DFE: gate-by-gate Trotter inverse, 2x gate count (not 6x)
  - ZNE: exponential fit at F>0.3, Richardson at F<0.3
  - Optimisation on FakeFez density-matrix sim (never touches real device)
  - Evaluation (DFE) on real device when USE_REAL_DEVICE=True
"""

import time
import signal
import numpy as np
import scipy.linalg as la
import networkx as nx
import warnings
from datetime import datetime
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.transpiler import CouplingMap

warnings.filterwarnings('ignore')


def ts():
    return datetime.now().strftime("[%H:%M:%S]")


# ── Timeout / partial-results guard ──────────────────────────────────
# At 9 minutes, prints whatever results exist so far, then lets the
# script continue (it won't be killed — just warned).
_RESULTS_SO_FAR = {}   # populated as each measurement completes
_START_TIME     = None
_QPU_BUDGET     = 9 * 60   # 9 minutes in seconds

def _save_partial_and_warn(signum, frame):
    elapsed = time.time() - _START_TIME
    print(f"\n{'!'*70}")
    print(f"{ts()} ⚠  {elapsed/60:.1f} min elapsed — 9-minute mark reached.")
    print(f"{ts()} Partial results so far:")
    if not _RESULTS_SO_FAR:
        print("  (nothing measured yet — still in local computation)")
    for k, v in _RESULTS_SO_FAR.items():
        print(f"  {k}: {v:.5f}")
    print(f"{ts()} Script continuing — any remaining jobs will still complete.")
    print(f"{'!'*70}\n")
    # Re-arm for another 60s warning if still running
    signal.alarm(60)

signal.signal(signal.SIGALRM, _save_partial_and_warn)


# ──────────── USER CONFIGURATION ────────────
USE_REAL_DEVICE   = True          # True → real IBM backend
REAL_BACKEND_NAME = 'ibm_fez'     # ibm_fez, ibm_kingston, or ibm_boston
SHOTS_FOR_DFE     = 2000
RUN_SEED          = 0

from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_ibm_runtime import SamplerV2 as IBMSampler
# ────────────────────────────────────────────

if USE_REAL_DEVICE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    token='0FtniVdfa-FkomX3aINnkCavYf1HIpWFtqO4GBVOz_jl'
)
    real_backend = service.backend(REAL_BACKEND_NAME)
    print(f"{ts()} Real backend: {real_backend.name}  "
          f"({real_backend.num_qubits}q, {real_backend.status().status_msg})")
else:
    real_backend = None

fake_backend_opt = FakeOptBackend()
print(f"{ts()} Fake backend (optimisation): {fake_backend_opt.backend_name}")

# ── Constants ──
N_QUBITS   = 8
DIM        = 2 ** N_QUBITS
MAX_ITERS  = 40
REG_LAMBDA = 1e-3

_X = np.array([[0, 1], [1, 0]], dtype=complex)
_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
_Z = np.array([[1, 0], [0, -1]], dtype=complex)
_PAIRS_2Q = {
    'XX': np.kron(_X, _X), 'XY': np.kron(_X, _Y), 'XZ': np.kron(_X, _Z),
    'YX': np.kron(_Y, _X), 'YY': np.kron(_Y, _Y), 'YZ': np.kron(_Y, _Z),
    'ZX': np.kron(_Z, _X), 'ZY': np.kron(_Z, _Y), 'ZZ': np.kron(_Z, _Z),
}
_NN      = ['XX', 'YY', 'ZZ']
N_PAIRS  = N_QUBITS - 1   # 7
N_TERMS  = N_PAIRS * 3    # 21
N_PARAMS = 2 * N_TERMS    # 42

_TERM_INFO = [
    {'P2q': _PAIRS_2Q[pn], 'qk_qa': N_QUBITS - 2 - pi, 'qk_qb': N_QUBITS - 1 - pi}
    for pi in range(N_PAIRS) for pn in _NN
]


# ── Path finding ──────────────────────────────────────────────────────
def find_quietest_path(backend):
    """Find the lowest-error 8-qubit chain on the given backend."""
    props = backend.properties()
    bg    = backend.configuration().basis_gates
    tq    = [g for g in bg if g in ['cx', 'cz', 'ecr']][0]
    G = nx.Graph()
    for q1, q2 in backend.configuration().coupling_map:
        try:    e1 = props.gate_error(tq, [q1, q2]) or 0.0
        except: e1 = 0.0
        try:    e2 = props.gate_error(tq, [q2, q1]) or 0.0
        except: e2 = 0.0
        try:    r1 = props.readout_error(q1) or 0.0
        except: r1 = 0.0
        try:    r2 = props.readout_error(q2) or 0.0
        except: r2 = 0.0
        G.add_edge(q1, q2, weight=(e1 + e2) / 2 + 0.1 * (r1 + r2) / 2)
    paths = []
    def dfs(n, p):
        if len(p) == 8: paths.append(p); return
        for nb in G.neighbors(n):
            if nb not in p: dfs(nb, p + [nb])
    for n in G.nodes: dfs(n, [n])
    scored = sorted(
        [(sum(G[p[i]][p[i+1]]['weight'] for i in range(7)), p) for p in paths]
    )
    # Print top 3 so you can see what the real device found
    print(f"{ts()} Top 3 quietest paths on {backend.backend_name}:")
    for i, (score, path) in enumerate(scored[:3]):
        errs = [G[path[j]][path[j+1]]['weight'] for j in range(7)]
        print(f"  #{i+1}: {path}  score={score:.5f}  mean_err={np.mean(errs):.4f}")
    return scored[0][1]


# Find quietest path on fake backend for optimisation
print(f"{ts()} Finding quietest path on FakeFez (for optimisation)...")
quietest_fake = find_quietest_path(fake_backend_opt)
print(f"{ts()} FakeFez quietest: {quietest_fake}")

# Find quietest path on real backend if needed
if USE_REAL_DEVICE:
    print(f"{ts()} Finding quietest path on {REAL_BACKEND_NAME} (for evaluation)...")
    quietest_real = find_quietest_path(real_backend)
    print(f"{ts()} Real quietest:   {quietest_real}")
else:
    quietest_real = quietest_fake


# ── Build simulators ──────────────────────────────────────────────────
def build_reduced_sim(backend, layout_8):
    """Build noise-matched DM and measurement simulators for an 8-qubit path."""
    p2v       = {p: i for i, p in enumerate(layout_8)}
    fc        = backend.configuration().coupling_map
    red_edges = [(p2v[a], p2v[b]) for a, b in fc
                 if a in layout_8 and b in layout_8]
    cmap      = CouplingMap(red_edges)
    fn        = NoiseModel.from_backend(backend)
    nm        = NoiseModel(basis_gates=backend.configuration().basis_gates)
    for phys_q in layout_8:
        vq = p2v[phys_q]
        try:
            ro = fn._local_readout_errors.get(phys_q)
            if ro: nm.add_readout_error(ro, [vq])
        except: pass
    for gn in backend.configuration().basis_gates:
        for qubits, error in fn._local_quantum_errors.get(gn, {}).items():
            nq = tuple(p2v[q] for q in qubits if q in layout_8)
            if len(nq) == len(qubits):
                nm.add_quantum_error(error, gn, nq)
    sim_dm   = AerSimulator(noise_model=nm, coupling_map=cmap, method='density_matrix')
    sim_meas = AerSimulator(noise_model=nm, coupling_map=cmap)
    return sim_dm, sim_meas, cmap


# Always build the FakeFez sim for optimisation (never uses real device)
_sim_dm, _sim_meas, _cm = build_reduced_sim(fake_backend_opt, quietest_fake)
basis_g_fake = fake_backend_opt.configuration().basis_gates

# For simulation-mode evaluation, also use FakeFez sims
# For real-device evaluation, the real backend is used directly in dfe_fidelity


# ── Target ────────────────────────────────────────────────────────────
def build_target(seed=0):
    """
    Build target as Trotter product of NN Pauli exponentials.
    psi_target from circuit statevector — guaranteed Qiskit little-endian.
    Returns (psi_target, coeffs, target_circuit).
    """
    rng    = np.random.default_rng(seed)
    coeffs = rng.uniform(-0.5, 0.5, N_TERMS)
    qc = QuantumCircuit(N_QUBITS)
    k  = 0
    for pi in range(N_PAIRS):
        for pn in _NN:
            qa = N_QUBITS - 2 - pi; qb = N_QUBITS - 1 - pi
            qc.unitary(la.expm(-1j * coeffs[k] * _PAIRS_2Q[pn]), [qa, qb]); k += 1
    sv_sim = AerSimulator(method='statevector')
    qc_sv  = qc.copy(); qc_sv.save_statevector()
    sv = sv_sim.run(
        transpile(qc_sv, sv_sim, optimization_level=0), shots=1
    ).result().data(0)['statevector']
    return np.array(sv), coeffs, qc


def build_target_inv_circuit(coeffs):
    """Gate-by-gate Trotter inverse. Exactly N_TERMS 2Q gates (same as ansatz)."""
    layers = []
    k = 0
    for pi in range(N_PAIRS):
        for pn in _NN:
            qa = N_QUBITS - 2 - pi; qb = N_QUBITS - 1 - pi
            layers.append((coeffs[k], _PAIRS_2Q[pn], qa, qb)); k += 1
    qc = QuantumCircuit(N_QUBITS)
    for c, P2q, qa, qb in reversed(layers):
        qc.unitary(la.expm(+1j * c * P2q), [qa, qb])
    return qc


def build_circuit(params):
    """Build parametric TOPAZ ansatz from (rho, tau) params."""
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    qc    = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
                   [t['qk_qa'], t['qk_qb']])
    return qc


# ── Objective (always uses FakeFez DM sim — no real device here) ──────
def _apply_gate(U, psi, qi, qj):
    N   = psi.ndim
    ax  = [qi, qj] + [k for k in range(N) if k not in (qi, qj)]
    psi = np.transpose(psi, ax).reshape(4, -1)
    psi = (U @ psi).reshape((2, 2) + (2,) * (N - 2))
    return np.transpose(psi, np.argsort(ax))


def compute_ideal_state(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6); rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    psi   = np.zeros((2,) * N_QUBITS, dtype=complex); psi[(0,) * N_QUBITS] = 1.0
    for j, t in enumerate(_TERM_INFO):
        psi = _apply_gate(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
                          psi, t['qk_qa'], t['qk_qb'])
    return psi.flatten()


def _reg_penalty(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6); rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]; total = 0.0
    for pi in range(N_PAIRS):
        B = np.zeros((4, 4), dtype=complex)
        for pp, pn in enumerate(_NN):
            j = pi * 3 + pp
            B += rho_n[j] * la.expm(-1j * rho_n[j] * tau[j] * _PAIRS_2Q[pn])
        dev    = B.conj().T @ B - np.eye(4, dtype=complex)
        total += np.real(np.trace(dev.conj().T @ dev))
    return total


def _noisy_dm(params, simulator):
    rho_w = np.maximum(params[:N_TERMS], 1e-6); rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    qc    = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']),
                   [t['qk_qa'], t['qk_qb']])
    qc_t = transpile(qc,
                     coupling_map=simulator.configuration().coupling_map,
                     basis_gates=basis_g_fake,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)
    qc_t.save_density_matrix()
    return np.array(simulator.run(qc_t, shots=1).result().data(0)['density_matrix'])


def noise_aware_objective(params, psi_target, simulator):
    psi_ideal = compute_ideal_state(params)
    rho_noisy = _noisy_dm(params, simulator)
    ideal_fid = float(np.abs(np.vdot(psi_target, psi_ideal)) ** 2)
    noisy_fid = float(np.clip(np.real(psi_target.conj() @ rho_noisy @ psi_target), 0, 1))
    loss      = 1.0 - noisy_fid + REG_LAMBDA * _reg_penalty(params)
    return loss, noisy_fid, ideal_fid


# ── MMA optimiser ─────────────────────────────────────────────────────
class MMAOptimizer:
    def __init__(self, n, move_limit=0.4, gamma=0.5):
        self.n = n; self.move_limit = move_limit; self.gamma = gamma
        self.L = self.U = self.prev_loss = None

    def init(self, p0, delta=0.6):
        self.L = p0 - delta; self.U = p0 + delta

    def step(self, x, grad):
        x_new = np.zeros_like(x)
        for i in range(self.n):
            g, xi, Li, Ui = grad[i], x[i], self.L[i], self.U[i]
            pi = abs(g) * (Ui - xi) ** 2 if g < 0 else 0.0
            qi = abs(g) * (xi - Li) ** 2 if g >= 0 else 0.0
            dn = pi / (Ui - xi + 1e-12) ** 2 + qi / (xi - Li + 1e-12) ** 2
            x_new[i] = (xi + (pi / (Ui - xi + 1e-12) - qi / (xi - Li + 1e-12)) / dn
                        if dn > 1e-12 else xi)
            x_new[i] = np.clip(x_new[i],
                               max(xi - self.move_limit, Li + 1e-6),
                               min(xi + self.move_limit, Ui - 1e-6))
        return x_new

    def update(self, x, loss):
        good   = self.prev_loss is None or loss < self.prev_loss - 1e-8
        s      = 1.2 / self.gamma if good else self.gamma
        self.L = x - s * (x - self.L); self.U = x + s * (self.U - x)
        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.prev_loss = loss


_prev_grad = None


def _hybrid_gradient(params, psi_target, simulator):
    global _prev_grad
    grad = np.zeros_like(params)
    for i in range(N_TERMS):
        pp = np.clip(params.copy(), 1e-6, None); pp[i] += 1e-4
        pm = np.clip(params.copy(), 1e-6, None); pm[i] -= 1e-4
        grad[i] = (noise_aware_objective(pp, psi_target, simulator)[0] -
                   noise_aware_objective(pm, psi_target, simulator)[0]) / 2e-4
    for i in range(N_TERMS, N_PARAMS):
        pp = params.copy(); pp[i] += np.pi / 4
        pm = params.copy(); pm[i] -= np.pi / 4
        grad[i] = (noise_aware_objective(pp, psi_target, simulator)[0] -
                   noise_aware_objective(pm, psi_target, simulator)[0]) / (np.pi / 2)
    if _prev_grad is None: _prev_grad = grad.copy(); return grad
    g = 0.4 * grad + 0.6 * _prev_grad; _prev_grad = g.copy(); return g


def run_dual_mma(init_params, psi_target, simulator):
    """
    MMA optimisation on the FakeFez density-matrix simulator.
    NEVER touches the real device. Returns best params found.
    """
    global _prev_grad; _prev_grad = None
    mma_r = MMAOptimizer(N_TERMS, move_limit=0.2)
    mma_r.init(init_params[:N_TERMS], delta=0.4)
    mma_t = MMAOptimizer(N_TERMS, move_limit=0.6)
    mma_t.init(init_params[N_TERMS:], delta=0.8)

    cur = init_params.copy()
    cur_loss, cur_fid, cur_ifid = noise_aware_objective(cur, psi_target, simulator)
    best_fid, best_params, stag = cur_fid, cur.copy(), 0
    print(f"{ts()} Init DM: {cur_fid:.4f}  (ideal {cur_ifid:.4f})")

    for it in range(MAX_ITERS):
        t0   = time.time()
        grad = _hybrid_gradient(cur, psi_target, simulator)
        new  = np.concatenate([mma_r.step(cur[:N_TERMS], grad[:N_TERMS]),
                                mma_t.step(cur[N_TERMS:], grad[N_TERMS:])])
        new_loss, new_fid, new_ifid = noise_aware_objective(new, psi_target, simulator)
        delta = new_fid - cur_fid

        if new_loss < cur_loss - 1e-6 or delta > -1e-5:
            cur, cur_loss, cur_fid, cur_ifid = new, new_loss, new_fid, new_ifid
            if cur_fid > best_fid:
                best_fid, best_params, stag = cur_fid, cur.copy(), 0
            else:
                stag += 1
            mma_r.move_limit = min(
                0.4, mma_r.move_limit * (
                    1.4 if delta > 0.01 else 1.2 if delta > 0.001 else 0.9))
        else:
            mma_r.move_limit = max(0.02, mma_r.move_limit * 0.8)
            stag += 1

        mma_t.move_limit = mma_r.move_limit * 2.0
        mma_r.update(cur[:N_TERMS], cur_loss)
        mma_t.update(cur[N_TERMS:], cur_loss)
        print(f"{ts()} Iter {it+1:2d}: DM={cur_fid:.4f} (ideal {cur_ifid:.4f}) "
              f"Δ={delta:+.4f} | {time.time()-t0:.1f}s | stag={stag}")
        if stag > 15:
            print(f"{ts()} Stagnated at iter {it+1}.")
            break

    print(f"{ts()} Optimisation done. Best DM fidelity: {best_fid:.4f}")
    return best_params


# ── Fidelity evaluation ───────────────────────────────────────────────
def exact_dm_fidelity(qc_ansatz, psi_target, dm_sim):
    """Exact fidelity via density matrix. Simulation only."""
    qc_t = transpile(qc_ansatz,
                     coupling_map=dm_sim.configuration().coupling_map,
                     basis_gates=basis_g_fake,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)
    qc_t.save_density_matrix()
    rho = np.array(dm_sim.run(qc_t, shots=1).result().data(0)['density_matrix'])
    return float(np.clip(np.real(psi_target.conj() @ rho @ psi_target), 0, 1))


def dfe_fidelity(qc_ansatz, target_inv_circuit, backend,
                 coupling_map, basis_gates, shots, label=""):
    """
    Direct Fidelity Estimation via P(|00..0>) after appending U†.
    Uses SamplerV2:
      - Real IBM backend  → IBMSampler(mode=backend)
      - AerSimulator      → AerSampler()
    Gate count = 2x ansatz (42 → 84 two-qubit gates).
    Chunked in 200-shot batches for live progress.
    """
    qc   = qc_ansatz.compose(target_inv_circuit)
    qc.measure_all()
    qc_t = transpile(qc,
                     coupling_map=coupling_map,
                     basis_gates=basis_gates,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)

    # Pick the right sampler based on backend type
    from qiskit_aer import AerSimulator as _AerSim
    if isinstance(backend, _AerSim):
        sampler = AerSampler()
    else:
        sampler = IBMSampler(mode=backend)   # real IBM device

    chunk = 200; total_zero = 0; total_shots = 0
    zero_str = '0' * N_QUBITS
    print(f"{ts()} [{label}] DFE ({shots} shots)")

    while total_shots < shots:
        this = min(chunk, shots - total_shots)
        job  = sampler.run([qc_t], shots=this)
        counts = job.result()[0].data.meas.get_counts()
        total_zero  += counts.get(zero_str, 0)
        total_shots += this
        print(f"{ts()} [{label}] {total_shots}/{shots}  "
              f"Fid={total_zero/total_shots:.5f}")

    return float(np.clip(total_zero / total_shots, 0, 1))


# ── ZNE ───────────────────────────────────────────────────────────────
def _fold(qc_t, sf):
    if sf == 1: return qc_t.copy()
    qi = qc_t.inverse(); f = qc_t.copy()
    for _ in range((sf - 1) // 2):
        f = f.compose(qi).compose(qc_t)
    return f


def _zne_method(exps):
    """Pick extrapolation method. Require strict monotonic decrease."""
    if not (exps[0] > exps[1] > exps[2]):
        return None, "non-monotonic — reporting λ=1 value"
    return ("exponential" if np.mean(exps) > 0.3
            else "richardson"), ("exponential fit" if np.mean(exps) > 0.3
                                 else "Richardson polynomial")


def _extrapolate(sfs, exps, method):
    if method == "richardson":
        return float(np.clip(
            np.polyval(np.polyfit(sfs, exps, deg=len(sfs) - 1), 0), 0, 1))
    try:
        log_e  = np.log(np.clip(exps, 1e-10, None))
        coeffs = np.polyfit(sfs, log_e, deg=1)
        return float(np.clip(np.exp(coeffs[1]), 0, 1))
    except:
        return exps[0]


def eval_zne_dm(qc_ansatz, psi_target, dm_sim, label=""):
    """ZNE with exact DM. Simulation only."""
    qc_t = transpile(qc_ansatz,
                     coupling_map=dm_sim.configuration().coupling_map,
                     basis_gates=basis_g_fake,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)
    sfs  = [1, 3, 5]; exps = []
    print(f"{ts()} [{label}] ZNE DM  λ={sfs}")
    for sf in sfs:
        qcf = _fold(qc_t, sf); qcf.save_density_matrix()
        rho = np.array(dm_sim.run(qcf, shots=1).result().data(0)['density_matrix'])
        fid = float(np.clip(np.real(psi_target.conj() @ rho @ psi_target), 0, 1))
        exps.append(fid); print(f"    λ={sf}: {fid:.5f}")
    method, desc = _zne_method(exps)
    if method:
        v = _extrapolate(sfs, exps, method)
        print(f"    → {v:.5f}  [{desc}]")
    else:
        v = exps[0]; print(f"    → {desc}: {v:.5f}")
    return v


def eval_zne_dfe(qc_ansatz, target_inv_circuit, meas_backend,
                 coupling_map, basis_gates, shots, label=""):
    """
    ZNE with DFE shot counts.
    Uses 2x shots per scale factor to keep shot noise below the ZNE signal.
    """
    zne_shots = max(shots * 2, 4000)
    qc_t = transpile(qc_ansatz,
                     coupling_map=coupling_map,
                     basis_gates=basis_gates,
                     initial_layout=list(range(N_QUBITS)),
                     optimization_level=1)
    sfs  = [1, 3, 5]; exps = []
    print(f"{ts()} [{label}] ZNE DFE  λ={sfs}  ({zne_shots} shots/scale)")
    for sf in sfs:
        fid = dfe_fidelity(_fold(qc_t, sf), target_inv_circuit, meas_backend,
                           coupling_map, basis_gates, zne_shots,
                           label=f"{label} λ={sf}")
        exps.append(fid)
    method, desc = _zne_method(exps)
    if method:
        v = _extrapolate(sfs, exps, method)
        print(f"{ts()} [{label}] ZNE → {v:.5f}  [{desc}]")
    else:
        v = exps[0]; print(f"{ts()} [{label}] {desc}: {v:.5f}")
    return v


# ══════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    mode = 'Real-Hardware' if USE_REAL_DEVICE else 'Simulation'
    print(f"\n{ts()} === TOPAZ {mode} — Quietest Path ===\n")

    # Start the 9-minute countdown (real-device mode only)
    if USE_REAL_DEVICE:
        _START_TIME = time.time()
        signal.alarm(_QPU_BUDGET)
        print(f"{ts()} ⏱  9-minute timer started. "
              f"Partial results will print at the 9-min mark.\n")

    # ── Target state ──
    psi_target, target_coeffs, qc_target = build_target(seed=RUN_SEED)
    target_inv_circ = build_target_inv_circuit(target_coeffs)

    # ── Baseline: random (unoptimised) params — same ansatz structure ──
    rng         = np.random.default_rng(RUN_SEED + 42)
    init_params = np.concatenate([rng.uniform(0.1, 0.4, N_TERMS),
                                  rng.uniform(0.1, np.pi, N_TERMS)])
    qc_baseline = build_circuit(init_params)

    print(f"{ts()} Target:   Trotter PTE circuit, seed={RUN_SEED}")
    print(f"{ts()} Baseline: same ansatz, random params (seed={RUN_SEED+42})")
    print(f"{ts()} TOPAZ:    MMA-optimised on FakeFez quietest path\n")

    # ── [Step 1] Optimisation — always on FakeFez, never real device ──
    # If saved params exist, load them (skip the ~5 hour optimisation).
    # Delete the .npy files to force a fresh optimisation run.
    import os
    if os.path.exists('best_params_quietest.npy'):
        best_params   = np.load('best_params_quietest.npy')
        init_params   = np.load('init_params.npy')
        target_coeffs_loaded = np.load('target_coeffs.npy')
        # Verify target coeffs match current seed (safety check)
        if np.allclose(target_coeffs_loaded, target_coeffs):
            print(f"{ts()} [Step 1/3] Loaded saved params (skipping optimisation).")
            print(f"{ts()} Delete best_params_quietest.npy to force re-optimisation.\n")
        else:
            print(f"{ts()} [Step 1/3] Saved params don't match current seed — re-optimising.")
            os.remove('best_params_quietest.npy')
            best_params = None
    else:
        best_params = None

    if best_params is None:
        print(f"{ts()} [Step 1/3] TOPAZ MMA optimisation (FakeFez DM sim)...")
        print(f"{ts()} Path: {quietest_fake}")
        t0 = time.time()
        best_params = run_dual_mma(init_params.copy(), psi_target, _sim_dm)
        print(f"{ts()} Optimisation took {time.time()-t0:.1f}s\n")
        np.save('best_params_quietest.npy', best_params)
        np.save('init_params.npy', init_params)
        np.save('target_coeffs.npy', target_coeffs)
        print(f"{ts()} Params saved: best_params_quietest.npy, "
              f"init_params.npy, target_coeffs.npy\n")

    qc_topaz = build_circuit(best_params)

    # ── [Step 2] Baseline and TOPAZ fidelity ─────────────────────────
    print(f"{ts()} [Step 2/3] Fidelity evaluation...")

    if USE_REAL_DEVICE:
        # Real device: DFE only, on the real quietest path
        print(f"{ts()} Evaluation path (real): {quietest_real}")
        p2v_real = {p: i for i, p in enumerate(quietest_real)}
        real_edges = [(p2v_real[a], p2v_real[b])
                      for a, b in real_backend.configuration().coupling_map
                      if a in quietest_real and b in quietest_real]
        real_cmap    = CouplingMap(real_edges)
        real_basis_g = real_backend.configuration().basis_gates

        # Also compute DM fidelity on FakeFez for reference
        base_dm_fake  = exact_dm_fidelity(qc_baseline, psi_target, _sim_dm)
        topaz_dm_fake = exact_dm_fidelity(qc_topaz,   psi_target, _sim_dm)
        _RESULTS_SO_FAR['baseline_DM_fakefez'] = base_dm_fake
        _RESULTS_SO_FAR['topaz_DM_fakefez']    = topaz_dm_fake
        print(f"  Baseline DM (FakeFez ref): {base_dm_fake:.4f}")
        print(f"  TOPAZ    DM (FakeFez ref): {topaz_dm_fake:.4f}  "
              f"(gain {topaz_dm_fake/max(base_dm_fake,1e-9):.2f}x)\n")

        print(f"{ts()} Running DFE on real {REAL_BACKEND_NAME}...")
        base_dfe  = dfe_fidelity(qc_baseline, target_inv_circ, real_backend,
                                 real_cmap, real_basis_g, SHOTS_FOR_DFE,
                                 label="base")
        _RESULTS_SO_FAR['baseline_DFE_real'] = base_dfe

        topaz_dfe = dfe_fidelity(qc_topaz, target_inv_circ, real_backend,
                                 real_cmap, real_basis_g, SHOTS_FOR_DFE,
                                 label="topaz")
        _RESULTS_SO_FAR['topaz_DFE_real'] = topaz_dfe
        _RESULTS_SO_FAR['gain_DFE_real']  = topaz_dfe / max(base_dfe, 1e-9)

    else:
        # Simulation: both DM (exact) and DFE (shot-based) for comparison
        print(f"{ts()} Evaluation path (FakeFez): {quietest_fake}")
        base_dm   = exact_dm_fidelity(qc_baseline, psi_target, _sim_dm)
        base_dfe  = dfe_fidelity(qc_baseline, target_inv_circ, _sim_meas,
                                 _cm, basis_g_fake, SHOTS_FOR_DFE, label="base")
        topaz_dm  = exact_dm_fidelity(qc_topaz, psi_target, _sim_dm)
        topaz_dfe = dfe_fidelity(qc_topaz, target_inv_circ, _sim_meas,
                                 _cm, basis_g_fake, SHOTS_FOR_DFE, label="topaz")

        print(f"\n  Baseline DM: {base_dm:.4f}  | DFE: {base_dfe:.4f}")
        print(f"  TOPAZ    DM: {topaz_dm:.4f}  | DFE: {topaz_dfe:.4f}  "
              f"(gain {topaz_dfe/max(base_dfe,1e-9):.2f}x)")

    # ── [Step 3] ZNE ─────────────────────────────────────────────────
    print(f"\n{ts()} [Step 3/3] ZNE evaluations...")

    if USE_REAL_DEVICE:
        # ZNE DM on FakeFez — runs locally, zero QPU cost
        base_zne_dm  = eval_zne_dm(qc_baseline, psi_target, _sim_dm,
                                    label="base (FakeFez)")
        _RESULTS_SO_FAR['baseline_ZNE_DM_fakefez'] = base_zne_dm
        topaz_zne_dm = eval_zne_dm(qc_topaz,   psi_target, _sim_dm,
                                    label="topaz (FakeFez)")
        _RESULTS_SO_FAR['topaz_ZNE_DM_fakefez'] = topaz_zne_dm
        # ZNE DFE on real device — SKIPPED to save QPU time
        # Set RUN_ZNE_ON_REAL = True if you have enough QPU minutes
        RUN_ZNE_ON_REAL = False
        if RUN_ZNE_ON_REAL:
            base_zne_dfe  = eval_zne_dfe(qc_baseline, target_inv_circ, real_backend,
                                          real_cmap, real_basis_g, SHOTS_FOR_DFE,
                                          label="base (real)")
            topaz_zne_dfe = eval_zne_dfe(qc_topaz, target_inv_circ, real_backend,
                                          real_cmap, real_basis_g, SHOTS_FOR_DFE,
                                          label="topaz (real)")
        else:
            print(f"{ts()} ZNE DFE on real device skipped (RUN_ZNE_ON_REAL=False).")
            print(f"{ts()} Raw DFE values are the primary real-hardware result.")
            base_zne_dfe  = None
            topaz_zne_dfe = None
    else:
        base_zne_dm   = eval_zne_dm(qc_baseline, psi_target, _sim_dm,  label="base")
        topaz_zne_dm  = eval_zne_dm(qc_topaz,   psi_target, _sim_dm,   label="topaz")
        base_zne_dfe  = eval_zne_dfe(qc_baseline, target_inv_circ, _sim_meas,
                                      _cm, basis_g_fake, SHOTS_FOR_DFE, label="base")
        topaz_zne_dfe = eval_zne_dfe(qc_topaz, target_inv_circ, _sim_meas,
                                      _cm, basis_g_fake, SHOTS_FOR_DFE, label="topaz")

    # ── Final summary ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print(f"{ts()} FINAL SUMMARY — Quietest 8-qubit path")
    print(f"  Baseline = same ansatz, random params  (seed {RUN_SEED+42})")
    print(f"  TOPAZ    = same ansatz, MMA-optimised  (FakeFez noise model)")
    if USE_REAL_DEVICE:
        print(f"  DM eval  = FakeFez simulation  (reference)")
        print(f"  DFE eval = real {REAL_BACKEND_NAME}")
    print("=" * 70)

    if USE_REAL_DEVICE:
        print(f"\n  FakeFez reference (DM, exact):")
        print(f"    Baseline DM : {base_dm_fake:.4f}  →  TOPAZ DM : {topaz_dm_fake:.4f}  "
              f"(gain {topaz_dm_fake/max(base_dm_fake,1e-9):.2f}x)")
        print(f"    ZNE base DM : {base_zne_dm:.4f}  →  ZNE TOPAZ: {topaz_zne_dm:.4f}  "
              f"(gain {topaz_zne_dm/max(base_zne_dm,1e-9):.2f}x)")
        print(f"\n  Real {REAL_BACKEND_NAME} (DFE, shot-based):")
        print(f"    Baseline DFE: {base_dfe:.4f} ±{np.sqrt(base_dfe*(1-base_dfe)/SHOTS_FOR_DFE):.4f}  "
              f"→  TOPAZ DFE: {topaz_dfe:.4f} ±{np.sqrt(topaz_dfe*(1-topaz_dfe)/SHOTS_FOR_DFE):.4f}  "
              f"(gain {topaz_dfe/max(base_dfe,1e-9):.2f}x)")
        if base_zne_dfe is not None:
            print(f"    ZNE base DFE: {base_zne_dfe:.4f}  →  ZNE TOPAZ: {topaz_zne_dfe:.4f}  "
                  f"(gain {topaz_zne_dfe/max(base_zne_dfe,1e-9):.2f}x)")
        else:
            print(f"    ZNE DFE: skipped (set RUN_ZNE_ON_REAL=True for full ZNE)")
    else:
        print(f"\n  DM (exact, FakeFez):")
        print(f"    Baseline: {base_dm:.4f}  →  TOPAZ: {topaz_dm:.4f}  "
              f"(gain {topaz_dm/max(base_dm,1e-9):.2f}x)")
        print(f"    ZNE base: {base_zne_dm:.4f}  →  ZNE TOPAZ: {topaz_zne_dm:.4f}  "
              f"(gain {topaz_zne_dm/max(base_zne_dm,1e-9):.2f}x)")
        print(f"\n  DFE (shot-based, FakeFez):")
        print(f"    Baseline: {base_dfe:.4f}  →  TOPAZ: {topaz_dfe:.4f}  "
              f"(gain {topaz_dfe/max(base_dfe,1e-9):.2f}x)")
        print(f"    ZNE base: {base_zne_dfe:.4f}  →  ZNE TOPAZ: {topaz_zne_dfe:.4f}  "
              f"(gain {topaz_zne_dfe/max(base_zne_dfe,1e-9):.2f}x)")

    print(f"\n  Noise floor: 1/{DIM} = {1/DIM:.4f}")
    print(f"  DFE is ~10-25% below DM (2x gate count is expected and correct).")
    print("=" * 70)


qiskit_runtime_service._discover_account:WARNING:2026-06-02 01:06:09,226: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-06-02 01:06:14,041: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-02 01:06:14,042: Using instance: open-instance, plan: open


[01:06:14] Real backend: ibm_fez  (156q, active)
[01:06:15] Fake backend (optimisation): fake_fez
[01:06:15] Finding quietest path on FakeFez (for optimisation)...
[01:06:16] Top 3 quietest paths on fake_fez:
  #1: [125, 124, 123, 136, 143, 142, 141, 140]  score=0.02457  mean_err=0.0035
  #2: [140, 141, 142, 143, 136, 123, 124, 125]  score=0.02457  mean_err=0.0035
  #3: [117, 125, 124, 123, 136, 143, 142, 141]  score=0.02478  mean_err=0.0035
[01:06:16] FakeFez quietest: [125, 124, 123, 136, 143, 142, 141, 140]
[01:06:16] Finding quietest path on ibm_fez (for evaluation)...
[01:06:16] Top 3 quietest paths on ibm_fez:
  #1: [127, 137, 147, 146, 145, 144, 143, 136]  score=0.02168  mean_err=0.0031
  #2: [136, 143, 144, 145, 146, 147, 137, 127]  score=0.02168  mean_err=0.0031
  #3: [127, 137, 147, 146, 145, 144, 143, 142]  score=0.02205  mean_err=0.0031
[01:06:16] Real quietest:   [127, 137, 147, 146, 145, 144, 143, 136]

[01:06:18] === TOPAZ Real-Hardware — Quietest Path ===

[01:06:18] ⏱ 